# Spatial-direction controls

The spatial PDE is refitted after episode-wise time reversal, and event lag planes are independently estimated for positive peaks and negative troughs across three cell samples.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from yumeow_plot import get_fig
ROOT = Path('..').resolve()
PDE = ROOT / 'logs/scripts/spatial_pde_library'
EVENTS = ROOT / 'logs/scripts/spatial_bin_event_propagation'
OUT = ROOT / 'figures/261027空间方向反证'
OUT.mkdir(parents=True, exist_ok=True)
forward_runs = ['initial', 'seed_20261010', 'seed_20261011']
reverse_runs = ['reversed_initial', 'reversed_seed_20261010', 'reversed_seed_20261011']
colors = {'Awake': '#cc5544', 'NREM': '#4477aa'}

In [2]:
fig, axes = get_fig(RN=1, CN=3, AW=4, AH=4, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(axes)
ax = next(ax_loader)
for direction, runs, marker, offset in [('forward', forward_runs, 'o', -.08), ('reversed', reverse_runs, 's', .08)]:
    tables = []
    for run in runs:
        table = pd.read_csv(PDE / run / 'model_summary.csv'); table['run'] = run; tables.append(table)
    table = pd.concat(tables)
    for index, state in enumerate(['Awake', 'NREM']):
        values = table[(table.state == state) & (table.model == 'local_advection_diffusion')].dx_r2_flat
        ax.scatter(np.full(len(values), index + offset), values, marker=marker, s=55, color=colors[state], label=direction if index == 0 else None)
ax.set_xticks([0, 1], ['Awake', 'NREM'])
ax.set(ylabel=r'Held-out $\Delta x$ $R^2$', title='PDE under time reversal')
ax.legend(frameon=False)
ax = next(ax_loader)
for direction, runs, alpha in [('forward', forward_runs, .7), ('reversed', reverse_runs, .35)]:
    for run in runs:
        c = pd.read_csv(PDE / run / 'coefficients.csv')
        for state in ['Awake', 'NREM']:
            part = c[(c.state == state) & (c.model == 'local_advection_diffusion')].set_index('term')
            x, y = part.loc['gradient_x'].coefficient_per_frame, part.loc['gradient_y'].coefficient_per_frame
            ax.arrow(0, 0, x, y, color=colors[state], alpha=alpha, width=.0008, length_includes_head=True)
ax.axhline(0, color='0.6', lw=.7); ax.axvline(0, color='0.6', lw=.7)
ax.set(xlabel=r'$\partial_x$ coefficient', ylabel=r'$\partial_y$ coefficient', title='Forward/reversed vectors', xlim=(-.11, .11), ylim=(-.11, .11))
ax.set_aspect('equal')
ax = next(ax_loader)
positive = ['initial', 'seed_20261010', 'seed_20261011']
negative = ['negative_initial', 'negative_seed_20261010', 'negative_seed_20261011']
for polarity, runs, marker, offset in [('positive', positive, 'o', -.08), ('negative', negative, 's', .08)]:
    for index, state in enumerate(['Awake', 'NREM']):
        angles = []
        for run in runs:
            d = pd.read_csv(EVENTS / run / 'event_plane_fits.csv')
            a = d[d.state == state].angle_radians.dropna().to_numpy()
            angles.append(np.degrees(np.angle(np.mean(np.exp(1j*a)))))
        ax.scatter(np.full(3, index + offset), angles, marker=marker, s=55, color=colors[state], label=polarity if index == 0 else None)
ax.set_xticks([0, 1], ['Awake', 'NREM'])
ax.set(ylabel='Circular-mean lag direction (deg)', title='Event-polarity control', ylim=(60, 145))
ax.legend(frameon=False)
fig.savefig(OUT / 'spatial_direction_controls.png')
fig.savefig(OUT / 'spatial_direction_controls.svg')
fig.savefig(OUT / 'spatial_direction_controls.pdf')
plt.show()

Time reversal does not destroy spatial predictability: reversed advection-diffusion R² is higher than forward in both states, and the fitted derivative vector approximately reverses direction. The PDE therefore describes robust spatiotemporal covariance but supplies no physical time arrow.

Negative events also show significant spatial lag planes. NREM trough directions (105°–117°) broadly agree with positive peaks (114°–129°). Awake troughs instead shift to 81°–92° from the positive 119°–128°, demonstrating polarity-dependent propagation geometry. A single fixed scan direction is therefore not a complete description, although acquisition timing remains a plausible contributor.